In [ ]:
import importlib
spec = importlib.util.find_spec('tensorflow')
if spec is None:
    print('tensorflow not available - skipping notebook')
else:
    import json
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    from maneuvers.data.loader import generate_synthetic_sequence
    from maneuvers.preprocessing import windowed_examples_from_sequence
    from maneuvers.models.cnn import _CNN1DMultiBranchClassifier

    # collect windows from multiple sequences
    X, y = [], []
    for seed in range(8):
        seq = generate_synthetic_sequence(duration_s=4.0, fs=50, seed=seed)
        XX, YY, _ = windowed_examples_from_sequence(seq, window_s=0.8, hop_s=0.4, fs=50)
        for xx, lbl in zip(XX, YY):
            if lbl != 'none':
                X.append(xx)
                y.append(0 if 'left' in lbl or 'right' in lbl else 1)
    X = np.asarray(X)
    y = np.asarray(y)

    # quick train/val split
    idx = np.arange(len(X))
    np.random.shuffle(idx)
    split = int(0.8 * len(X))
    train_idx, val_idx = idx[:split], idx[split:]

    model = _CNN1DMultiBranchClassifier(seq_len=X.shape[1], n_accel=3, n_gyro=3, n_classes=2, epochs=6, batch_size=8)
    history = model.model if False else None
    # wrap validation data as tuples accepted by Keras
    model.fit(X[train_idx], y[train_idx], validation_data=(X[val_idx], y[val_idx]))

    # training history plot (available via model.model.history.history)
    hist = model.model.history.history
    plt.figure(figsize=(6, 3))
    plt.plot(hist['loss'], label='train_loss')
    plt.plot(hist['val_loss'], label='val_loss')
    plt.legend()
    plt.title('Training loss')
    plt.tight_layout()
    plt.show()